To read:
https://towardsdatascience.com/how-to-download-and-visualize-your-twitter-network-f009dbbf107b
https://benjaminstrick.com/how-i-scrape-and-analyse-twitter-networks/
https://datadistillery.digital/portfolio/twitter-data-pipeline/


# Global variables to change

In [13]:
words = "#digitalebildung" # The words you want tweets to include
query = words + " (is:retweet OR is:quote) lang:de" # The query you want to search for
tweets_limit = 1000 # The number of tweets you want to return in total (must be divisible by 100 and cannot exceed 2000)

# Prepare environment

In [2]:
# Import packages
import os
import tweepy
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import networkx as nx
import plotly.graph_objects as go
from config import bearer_token

# Initialize the Tweepy client
client = tweepy.Client(bearer_token=bearer_token)

In [4]:
# Confirm the client is initialized by printing the 10 most recent tweets using your words
for tweet in client.search_recent_tweets(words).data:
    print(tweet.text)

RT @chu_tum: Aufgepasst! Bald geht unser digitaler Selbstlernkurs zu #FlippedClassroom, #KritischesDenken und #Hausaufgaben online ➡️ https…
Jeden zweiten Mittwoch eine neue Folge „Einig uneinig – Digitale Bildung to be discussed“. 
In der ersten Folge spricht unsere Kanzlerin mit Karen Hoyndorf über das Thema „Zerstört Weiterbildung die effektive Arbeitszeit?“.
https://t.co/VgtnppLaOf
#Podcast #DigitaleBildung
RT @helliwood: Zu #KI arbeiten wir ja schon eine ganze Weile, heute eine kleine Einführung zu #ChatGPT von @Wapoid #futureskills #digitaleB…
Zu #KI arbeiten wir ja schon eine ganze Weile, heute eine kleine Einführung zu #ChatGPT von @Wapoid #futureskills #digitaleBildung #Medienpädagogik #twlz https://t.co/hw9PxcVcZd
Wie werden sich die #Medien entwickeln? 
 ________ 
 #Massenmedien #Bundestag #Digitalisierung #Muenchen #Berlin #DigitaleBildung #Frankfurt #Digitalisierung #Augsburg #Arbeitsmarkt 
 https://t.co/o6hFcUZcSx
RT @KICampus: Letzter Aufruf: Noch bis zum 31. Januar läuf

# Download Tweets

Fields available: https://developer.twitter.com/en/docs/twitter-api/fields

A guide: https://dev.to/twitterdev/a-comprehensive-guide-for-using-the-twitter-api-v2-using-tweepy-in-python-15d9

Build a query: https://developer.twitter.com/en/docs/twitter-api/tweets/search/integrate/build-a-query

In [3]:
# Define a function to query tweets
def get_retweets(query, num_tweets=1000):
    # Initialize an empty DataFrame to store user data and tweets
    tweets_df = pd.DataFrame()
    
    # Return the number of batches based on num_tweets
    if num_tweets > 2000:
        raise ValueError("`num_tweets` must be less than or equal to 2000.")
    elif num_tweets % 100 != 0:
        raise ValueError("`num_tweets` must be a multiple of 100.")
    max_results = 100
    limit = num_tweets / max_results

    # Iterate through batches of tweets
    for tweet_batch in tweepy.Paginator(client.search_recent_tweets, 
                                query=query,
                                expansions='author_id',
                                user_fields=["username", "id"],
                                tweet_fields=["public_metrics"],
                                max_results=100,
                                limit=limit
                                ):
        batch_data = pd.DataFrame(tweet_batch.data)
        users = {u["id"]: u["username"] for u in tweet_batch.includes["users"]}
        batch_data["retweeter"] = batch_data["author_id"].map(users)
        # Concatenate temporary DataFrames to existing DataFrames
        tweets_df = pd.concat([tweets_df, batch_data])

    # Merge user information to tweet information on author_id
    # Extract original tweeter from tweet text
    tweets_df["tweeter"] = tweets_df["text"].str.extract(r"@(\w+)")
    # Return DataFrame
    return tweets_df


# Create a DataFrame using the function defined above
df = get_retweets(query, tweets_limit)

# Preview the DataFrame
df

,author_id,edit_history_tweet_ids,id,public_metrics,text,retweeter,tweeter
0,19485870,[1618389685415575554],1618389685415575554,"{'retweet_count': 2, 'reply_count': 0, 'like_c...",RT @thomas_dettling: ⬛️ „Wenn Deutschland im i...,AkwyZ,thomas_dettling
1,37956761,[1618381764585738241],1618381764585738241,"{'retweet_count': 2, 'reply_count': 0, 'like_c...",RT @thomas_dettling: ⬛️ „Wenn Deutschland im i...,MartinGaedt,thomas_dettling
2,2353159116,[1618375435603283973],1618375435603283973,"{'retweet_count': 3, 'reply_count': 0, 'like_c...",RT @FloRa_Education: Gerade findet der Kick-of...,ArneKlauke,FloRa_Education
3,1367088815928729611,[1618354162907111424],1618354162907111424,"{'retweet_count': 3, 'reply_count': 0, 'like_c...",RT @KICampus: Letzter Aufruf: Noch bis zum 31....,HeiCAD_HHU,KICampus
4,564870153,[1618321418323906562],1618321418323906562,"{'retweet_count': 4, 'reply_count': 0, 'like_c...",RT @chu_tum: Aufgepasst! Bald geht unser digit...,ConradiConradi,chu_tum
...,...,...,...,...,...,...,...
25,1160183215308623873,[1616042818765553664],1616042818765553664,"{'retweet_count': 55, 'reply_count': 0, 'like_...",RT @werkstatt_bpb: #ChatGPT in der #Schule - (...,Bot_TwLehrerZ,werkstatt_bpb
26,18971283,[1616029301513535488],1616029301513535488,"{'retweet_count': 4, 'reply_count': 0, 'like_c...",RT @KICampus: Studiengänge für die digitale We...,stefangoellner,KICampus
27,1032569317483859968,[1616027698668658689],1616027698668658689,"{'retweet_count': 4, 'reply_count': 0, 'like_c...",RT @KICampus: Studiengänge für die digitale We...,twcampus_bot,KICampus
28,2225298907,[1616027025491345409],1616027025491345409,"{'retweet_count': 4, 'reply_count': 0, 'like_c...",RT @KICampus: Studiengänge für die digitale We...,lucasllaux,KICampus


Save the dataframe

In [5]:
df.to_parquet("data/tweets.parquet", index=False)

# Calculating the Importance of Users in the Network
The next step is to analyze the network. The extracted tweets and relevant information are returned as a DataFrame containing an 'edge list', or a list of all edges between nodes. The code then calculates three measures of centrality for each user in the network.

- **In-degree centrality** represents the number of edges going into a node. In the case of retweets, centrality will indicate that a user is getting a large number of retweets.
- **Out-degree centrality** represents the number of edges going out of a node. In the case of retweets, centrality will indicate that a user is retweeting a lot.
- **Betweeness centrality**  represents the number of 'shortest paths' between nodes that pass through through a specific node. In the case of tweets, it measures the extent to which a user connects other communities of users.

The code also stores the number of in_degrees (tweets) and out_degrees (retweets) to visualize later with Plotly.

In [6]:
df = pd.read_parquet("data/tweets.parquet")

In [7]:
# Create a directed network graph from the DataFrame
G = nx.from_pandas_edgelist(df, "retweeter", "tweeter", create_using=nx.DiGraph())

# Calculate the in-degree centrality for retweets
in_centrality = nx.in_degree_centrality(G)

# Caluculate the total number of tweets for later plotting purposes
in_degrees = dict(G.in_degree())

# Store centralities in DataFrame
popular_tweeters = pd.DataFrame(
    list(in_centrality.items()), columns=["username", "in-degree_centrality"]
)

# Print the most important users by their centrality
popular_tweeters.sort_values("in-degree_centrality", ascending=False).head()

,username,in-degree_centrality
30,werkstatt_bpb,0.393701
58,DBmS_de,0.062992
6,KICampus,0.062992
55,NumiScience,0.055118
27,hoou_haw,0.055118


In [8]:
# Calculate the out-degree centrality for retweets
out_centrality = nx.out_degree_centrality(G)

# Caluculate the total number of retweets for later plotting purposes
out_degrees = dict(G.out_degree())

# Store centralities in DataFrame
active_retweeters = pd.DataFrame(
    list(out_centrality.items()), columns=["username", "out-degree_centrality"]
)

# Print the most important users by the amount they retweet
active_retweeters.sort_values("out-degree_centrality", ascending=False).head()

,username,out-degree_centrality
9,Bot_TwLehrerZ,0.086614
36,digibits_de,0.015748
48,elerncoach,0.015748
40,FachportalPaed,0.015748
18,mike_bernd_,0.015748


In [9]:
# Create an undirected network graph from the DataFrame
G = nx.from_pandas_edgelist(df, "retweeter", "tweeter")

# Calculate the betweenness centrality for retweets
betweetnness = nx.betweenness_centrality(G)

# Store centralities in DataFrame
bridging_users = pd.DataFrame(
    list(betweetnness.items()), columns=["username", "betweenness"]
)

# Print the most important users by how much they bridge other users
bridging_users.sort_values("betweenness", ascending=False).head()

,username,betweenness
30,werkstatt_bpb,0.404949
9,Bot_TwLehrerZ,0.214848
36,digibits_de,0.168479
6,KICampus,0.128984
8,chu_tum,0.062742


# Visualizing Follower Networks
The code below creates an interactive network visualization using Plotly. There are a number of different attributes to the plot worth noting:
- The nodes are colored by the number of retweeted tweets. Those with a number of retweeted tweets are colored blue and purple, and those who only retweet are colored yellow.
- Those with more connections in total are larger to make it easier to spot them. 

In [14]:
# Create the graph and specify the layout of the graph
G = nx.from_pandas_edgelist(df, "retweeter", "tweeter")
pos = nx.drawing.layout.spring_layout(G)
nx.set_node_attributes(G, pos, "pos")

# Create a dictionary of nodes and their order
nodes_dict = {id: node for (id, node) in enumerate(G.nodes())}

# Gather edge positions for visualization
edge_x = []
edge_y = []
for edge in G.edges():
    x0_point, y0_point = G.nodes[edge[0]]["pos"]
    x1_point, y1_point = G.nodes[edge[1]]["pos"]
    edge_x.append(x0_point)
    edge_x.append(x1_point)
    edge_x.append(None)
    edge_y.append(y0_point)
    edge_y.append(y1_point)
    edge_y.append(None)

# Add edges as disconnected lines
edge_trace = go.Scatter(
    x=edge_x,
    y=edge_y,
    line=dict(width=0.5, color="#000000"),
    hoverinfo="none",
    mode="lines",
)

# Gather node positions for visualization
node_x = []
node_y = []
for node in G.nodes():
    x, y = G.nodes[node]["pos"]
    node_x.append(x)
    node_y.append(y)

# Iterate through the nodes and store the usernames and tweeting information
node_text = []
node_adjacencies = []
node_sizes = []
node_colors = []

# Iterate through the nodes and create the text to be shown on hover
for node_number, adjacencies in enumerate(G.adjacency()):
    node_text.append(  # Set the text to be shown on hover
        "Username: "
        + str(adjacencies[0])  # The username
        + "<br>"
        + "Number of Connections: "
        + str(len(adjacencies[1]))  # The number of connections
        + "<br>"
        + "Number of Retweeted Tweets: "
        + str(in_degrees[nodes_dict[node_number]])
        + "<br>"
        + "Number of Retweets: "
        + str(out_degrees[nodes_dict[node_number]])
    )
    node_adjacencies.append(len(adjacencies[1]))
    node_sizes.append(len(adjacencies[1]))
    node_colors.append(in_degrees[nodes_dict[node_number]])

# Log transform the color list for visualization purposes
node_colors = np.array(node_colors)
node_colors_temp = np.where(node_colors > 1.0e-10, node_colors, 1.0e-10)
node_colors_log = np.log10(node_colors_temp)

# Scale the size of the nodes between two values
scaler = MinMaxScaler(feature_range=(5, 30))
node_sizes = scaler.fit_transform(np.array(node_sizes).reshape(-1, 1))

# Plot the nodes
node_trace = go.Scatter(
    x=node_x,
    y=node_y,
    mode="markers",
    hoverinfo="text",
    marker=dict(
        showscale=True,
        colorscale="Plasma",  # For more colorscale options, go here: https://plotly.com/python/builtin-colorscales/
        reversescale=True,
        color=[],
        size=10,
        colorbar=dict(
            thickness=15,
            title="Number of Retweeted Tweets (Log Transformed)",
            titlefont=dict(size=12),
            xanchor="left",
            titleside="right",
            tickvals=[min(node_colors_log), max(node_colors_log)],
            ticktext=["Low", "High"],
        ),
        line=dict(color="Black", width=0.5),
    ),
)

# Set the size, color, and text of the nodes
node_trace.marker.size = node_sizes
node_trace.marker.color = node_colors_log
node_trace.text = node_text

# Customize and display the figure
fig = go.Figure(
    data=[edge_trace, node_trace],
    layout=go.Layout(
        title="<b>Retweet Network Graph for "
        + str(words)
        + "<b>",  # Set your title here
        title_x=0.5,
        titlefont_size=16,
        showlegend=False,
        paper_bgcolor="#e6e6e6",  # Set the background color (excluding the plot) here
        plot_bgcolor="#e6e6e6",  # Set the plot background color here
        font_color="#000000",
        hovermode="closest",
        margin=dict(b=20, l=5, r=5, t=40),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        width=800,  # Adjust the width of the plot
        height=500,  # Adjust the height of the plot
    ),
)
fig.show()